In [8]:
# ── Dependency Guard ───────────────────────────────────────────────
import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'requests': 'requests',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import requests                  # Ergast/Jolpica API access

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')

All required packages already installed ✓
pandas  : 2.3.3
numpy   : 2.4.3


In [9]:
# ── Reproducibility Header ────────────────────────────────────────────

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.12.3
NumPy   : 2.4.3
Seed    : 414


In [10]:
# ── Rule-Based Baseline using the same extracted data as EDA.ipynb ─────────

# Same data extraction used in EDA.ipynb
def get_season_results(year: int) -> pd.DataFrame:
    offset = 0
    limit = 100
    rows = []

    while True:
        url = f"https://api.jolpi.ca/ergast/f1/{year}/results.json?limit={limit}&offset={offset}"
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        data = response.json()['MRData']
        total = int(data['total'])
        races = data['RaceTable']['Races']

        for race in races:
            for result in race['Results']:
                rows.append({
                    'season': int(race['season']),
                    'round': int(race['round']),
                    'race_name': race['raceName'],
                    'driver': result['Driver']['driverId'],
                    'driver_name': f"{result['Driver']['givenName']} {result['Driver']['familyName']}",
                    'constructor': result['Constructor']['constructorId'],
                    'grid': int(result['grid']),
                    'position': int(result['position']) if result['position'].isdigit() else None,
                    'status': result['status']
                })

        offset += limit
        if offset >= total:
            break

    return pd.DataFrame(rows)

# Same seasons and feature engineering as EDA
seasons = [2023, 2024]
dfs = []
for year in seasons:
    df_year = get_season_results(year)
    print(f"{year}: {len(df_year)} rows, {df_year['round'].nunique()} races")
    dfs.append(df_year)

df = pd.concat(dfs, ignore_index=True)
df['top10'] = (df['position'] <= 10)

# Same temporal split used in EDA.ipynb
train = df[df['season'] == 2023]
val = df[(df['season'] == 2024) & (df['round'] <= 10)]
test = df[(df['season'] == 2024) & (df['round'] > 10)]

# Baseline evaluation on validation split
predictions_df = val.copy()
predictions_df = predictions_df[predictions_df['grid'].notna()].copy()
predictions_df['predicted_top10'] = predictions_df['grid'] <= 10
predictions_df['actual_top10'] = predictions_df['top10'].astype(bool)
predictions_df['correct'] = predictions_df['predicted_top10'] == predictions_df['actual_top10']

# Calculate metrics
total_predictions = len(predictions_df)
correct_predictions = predictions_df['correct'].sum()
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

from sklearn.metrics import precision_score, f1_score
if total_predictions > 0:
    precision = precision_score(
        predictions_df['actual_top10'],
        predictions_df['predicted_top10'],
        zero_division=0
    )
    f1 = f1_score(
        predictions_df['actual_top10'],
        predictions_df['predicted_top10'],
        zero_division=0
    )
else:
    precision = 0.0
    f1 = 0.0

print("=" * 70)
print("\n📊 RULE-BASED PREDICTION RESULTS (EDA-aligned data)")
print(f"{'─' * 70}")
print("Data source: Ergast/Jolpica extraction (same as EDA.ipynb)")
print("Evaluation split: 2024 rounds 1-10 (validation)")
print("Rule: If grid <= 10 → Predict Top-10 | Otherwise → Predict Not Top-10")
print(f"{'─' * 70}")
print(f"Total Predictions:  {total_predictions}")
print(f"Correct:            {correct_predictions}")
print(f"Incorrect:          {total_predictions - correct_predictions}")
print(f"\n🎯 ACCURACY:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"🎯 PRECISION: {precision:.4f} ({precision*100:.2f}%)")
print(f"🎯 F1-SCORE:  {f1:.4f} ({f1*100:.2f}%)")
print(f"{'─' * 70}")

print("\nDetailed Breakdown:")
print(f"  - Correctly predicted Top-10:      {predictions_df[(predictions_df['predicted_top10']) & (predictions_df['correct'])].shape[0]}")
print(f"  - Correctly predicted Not Top-10:  {predictions_df[(~predictions_df['predicted_top10']) & (predictions_df['correct'])].shape[0]}")
print(f"  - False positives (Top-10):        {predictions_df[(predictions_df['predicted_top10']) & (~predictions_df['correct'])].shape[0]}")
print(f"  - False negatives (Top-10):        {predictions_df[(~predictions_df['predicted_top10']) & (~predictions_df['correct'])].shape[0]}")


2023: 440 rows, 22 races
2024: 479 rows, 24 races

📊 RULE-BASED PREDICTION RESULTS (EDA-aligned data)
──────────────────────────────────────────────────────────────────────
Data source: Ergast/Jolpica extraction (same as EDA.ipynb)
Evaluation split: 2024 rounds 1-10 (validation)
Rule: If grid <= 10 → Predict Top-10 | Otherwise → Predict Not Top-10
──────────────────────────────────────────────────────────────────────
Total Predictions:  199
Correct:            173
Incorrect:          26

🎯 ACCURACY:  0.8693 (86.93%)
🎯 PRECISION: 0.8700 (87.00%)
🎯 F1-SCORE:  0.8700 (87.00%)
──────────────────────────────────────────────────────────────────────

Detailed Breakdown:
  - Correctly predicted Top-10:      87
  - Correctly predicted Not Top-10:  86
  - False positives (Top-10):        13
  - False negatives (Top-10):        13


This baseline is now aligned with the required Top-10 target (instead of DNF), and its accuracy is the reference to improve in Lab 2.